<a href="https://colab.research.google.com/github/Liping-LZ/BDAO_DSDO/blob/main/Week%201/02_BDAO_Block1_Workshop_FakeStore_ANSWERS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BDAO Block 1 — API Workshop: FakeStore API

**FakeStore API:** https://fakestoreapi.com  
A free, open ecommerce API — no authentication needed.

---
## How this workshop works

| Stage | What students do |
|---|---|
| **Explore** | Call the API, understand what data is available |
| **Analyse** | Extract insights from the data |
| **Connect** | Combine data from two endpoints to answer a business question |

---

## Setup — run this first

In [ ]:
import requests
import json
import pandas as pd

pd.set_option('display.max_columns', 12)
pd.set_option('display.max_colwidth', 45)
pd.set_option('display.float_format', '{:.2f}'.format)

BASE = 'https://fakestoreapi.com'

print('Ready. Base URL:', BASE)
print()
print('Documentation: https://fakestoreapi.com/docs')

Ready. Base URL: https://fakestoreapi.com

Documentation: https://fakestoreapi.com/docs


---
## Stage 1 — EXPLORE: Understand the API

### Task 1.1 — What does the products endpoint return?

In [ ]:
# Call the products endpoint
response = requests.get(f'{BASE}/products')
products = response.json()

print(f'Status code: {response.status_code}')
print(f'Number of products returned: {len(products)}')
print()
print('First product (raw JSON):')
print(json.dumps(products[0], indent=2))

Status code: 200
Number of products returned: 20

First product (raw JSON):
{
  "id": 1,
  "title": "Fjallraven - Foldsack No. 1 Backpack, Fits 15 Laptops",
  "price": 109.95,
  "description": "Your perfect pack for everyday use and walks in the forest. Stash your laptop (up to 15 inches) in the padded sleeve, your everyday",
  "category": "men's clothing",
  "image": "https://fakestoreapi.com/img/81fPKd-2AYL._AC_SL1500_t.png",
  "rating": {
    "rate": 3.9,
    "count": 120
  }
}


In [ ]:
# What fields does each product have?
print('Fields in each product:')
for key, value in products[0].items():
    print(f'  {key:<15} → {type(value).__name__:<10}')

Fields in each product:
  id              → int       
  title           → str       
  price           → float     
  description     → str       
  category        → str       
  image           → str       
  rating          → dict      


### Task 1.2 — What categories exist?

**Business question:** how many product categories does this store have?

In [ ]:
# ANSWER
categories = requests.get(f'{BASE}/products/categories').json()

print(f'Number of categories: {len(categories)}')
print()
print('Available categories:')
for cat in categories:
    print(cat)

Number of categories: 4

Available categories:
electronics
jewelery
men's clothing
women's clothing


**Expected output:**
```
Number of categories: 4

Available categories:
  electronics
  jewelery
  men's clothing
  women's clothing
```

### Task 1.3 — Get products from a specific category

**Business question:** what products does this store sell in the electronics category?

In [ ]:
# ANSWER
electronics = requests.get(f'{BASE}/products/category/electronics').json()

print(f'Electronics products: {len(electronics)}')
print()
print('Products:')
for p in electronics:
    print(f"  £{p['price']:.2f}  {p['title'][:55]}") # fomatting price with two decimal, title with first 55 characters

Electronics products: 6

Products:
  £64.00  WD 2TB Elements Portable External Hard Drive - USB 3.0 
  £109.00  SanDisk SSD PLUS 1TB Internal SSD - SATA III 6 Gb/s
  £109.00  Silicon Power 256GB SSD 3D NAND A55 SLC Cache Performan
  £114.00  WD 4TB Gaming Drive Works with Playstation 4 Portable E
  £599.00  Acer SB220Q bi 21.5 inches Full HD (1920 x 1080) IPS Ul
  £999.99  Samsung 49-Inch CHG90 144Hz Curved Gaming Monitor (LC49


---
## Stage 2 — ANALYSE: Extract business insights (simple)

### Task 2.1 — Build a product catalogue DataFrame

**Business question:** give the buying team a clean summary of all products with prices and ratings.

In [ ]:
# ANSWER
# rating is nested — p['rating']['rate'] not p['rating_rate']

records = []
for p in products:
    records.append({
        'id':           p['id'],
        'title':        p['title'][:45],
        'price':        p['price'],
        'category':     p['category'],
        'rating_score': p['rating']['rate'],    # nested dict access
        'rating_count': p['rating']['count']    # nested dict access
    })

df_products = pd.DataFrame(records)
print(f'Product catalogue: {len(df_products)} products')
df_products

Product catalogue: 20 products


,id,title,price,category,rating_score,rating_count
0,1,"Fjallraven - Foldsack No. 1 Backpack, Fit...",109.95,men's clothing,3.90,120
1,2,Mens Casual Premium Slim Fit T-Shirts,22.30,men's clothing,4.10,259
2,3,Mens Cotton Jacket,55.99,men's clothing,4.70,500
3,4,Mens Casual Slim Fit,15.99,men's clothing,2.10,430
4,5,John Hardy Women's Legends Naga Gold & Si...,695.00,jewelery,4.60,400
5,6,Solid Gold Petite Micropave,168.00,jewelery,3.90,70
6,7,White Gold Plated Princess,9.99,jewelery,3.00,400
7,8,Pierced Owl Rose Gold Plated Stainless St...,10.99,jewelery,1.90,100
8,9,WD 2TB Elements Portable External Hard Dr...,64.00,electronics,3.30,203
9,10,SanDisk SSD PLUS 1TB Internal SSD - SATA ...,109.00,electronics,2.90,470


### Task 2.2 — Which category has the highest average price?

**Business question:** which category commands the highest prices on average?

In [ ]:
# ANSWER
category_prices = (
    df_products
    .groupby('category')
    .agg(
        product_count = ('id', 'count'),
        avg_price     = ('price', 'mean'),
        avg_rating    = ('rating_score', 'mean')
    )
    .round(2)
    .sort_values('avg_price', ascending=False)
)

print('Average price by category:')
category_prices

Average price by category:


,product_count,avg_price,avg_rating
category,,,
electronics,6,332.50,3.48
jewelery,4,221.00,3.35
men's clothing,4,51.06,3.70
women's clothing,6,26.29,3.68


**Expected result:**  
Electronics has the highest average price, followed by jewellery.  
Clothing categories are significantly cheaper.

### Task 2.3 — Find the best value products

**Business question:** high ratings + low prices = best value for marketing to promote.

In [ ]:
# ANSWER
median_price = df_products['price'].median()
print(f'Median price: £{median_price:.2f}')
print()

best_value = (
    df_products[
        (df_products['rating_score'] > 4.0) &
        (df_products['price'] < median_price)
    ]
    .sort_values('rating_score', ascending=False)
)

print(f'Best value products (rating > 4.0, price below median £{median_price:.2f}):')
best_value[['title','category','price','rating_score','rating_count']]

Median price: £56.49

Best value products (rating > 4.0, price below median £56.49):


,title,category,price,rating_score,rating_count
2,Mens Cotton Jacket,men's clothing,55.99,4.70,500
17,MBJ Women's Solid Short Sleeve Boat Neck V,women's clothing,9.85,4.70,130
18,Opna Women's Short Sleeve Moisture,women's clothing,7.95,4.50,146
1,Mens Casual Premium Slim Fit T-Shirts,men's clothing,22.30,4.10,259


**Notes:**  
Common mistake: using `AND` instead of `&` for combining conditions, or forgetting the parentheses around each condition.  
Correct: `(condition1) & (condition2)`  
Wrong: `condition1 & condition2` (can produce unexpected results) or `condition1 AND condition2` (SyntaxError)

---
## Stage 3 — CONNECT: Combine two API endpoints

### Task 3.1 — Understand the carts endpoint

In [ ]:
# Call the carts endpoint
carts = requests.get(f'{BASE}/carts').json()

print(f'Number of carts (orders): {len(carts)}')
print()
print('First cart (raw):')
print(json.dumps(carts[0], indent=2))

Number of carts (orders): 7

First cart (raw):
{
  "id": 1,
  "userId": 1,
  "date": "2020-03-02T00:00:00.000Z",
  "products": [
    {
      "productId": 1,
      "quantity": 4
    },
    {
      "productId": 2,
      "quantity": 1
    },
    {
      "productId": 3,
      "quantity": 6
    }
  ],
  "__v": 0
}


In [ ]:
# Build order lines table — unpack products from each cart
order_lines = []
for cart in carts:
    for item in cart['products']:
        order_lines.append({
            'cart_id':    cart['id'],
            'user_id':    cart['userId'],
            'date':       cart['date'],
            'product_id': item['productId'],
            'quantity':   item['quantity']
        })

df_orders = pd.DataFrame(order_lines)
print(f'Order lines: {len(df_orders)} (one row per product per cart)')
df_orders.head()

Order lines: 14 (one row per product per cart)


,cart_id,user_id,date,product_id,quantity
0,1,1,2020-03-02T00:00:00.000Z,1,4
1,1,1,2020-03-02T00:00:00.000Z,2,1
2,1,1,2020-03-02T00:00:00.000Z,3,6
3,2,1,2020-01-02T00:00:00.000Z,2,4
4,2,1,2020-01-02T00:00:00.000Z,1,10


**Notes:**  
The nested loop here is the key concept — each cart contains a list of products.  
Students need to loop over carts AND loop over items within each cart.  
If confused, ask them to print `carts[0]['products']` and look at the structure first.

### Task 3.2 — Join orders to products

**Business question:** which categories generate the most sales?

In [ ]:
# ANSWER
# Join on: df_orders['product_id'] = df_products['id']

df_joined = pd.merge(
    df_orders,
    df_products[['id','title','category','price']],
    left_on  = 'product_id',
    right_on = 'id',
    how      = 'left'
)

# Revenue = price × quantity
df_joined['revenue'] = df_joined['price'] * df_joined['quantity']

print(f'Joined order lines: {len(df_joined)}')
df_joined[['cart_id','product_id','title','category','quantity','price','revenue']].head(8)

Joined order lines: 14


,cart_id,product_id,title,category,quantity,price,revenue
0,1,1,"Fjallraven - Foldsack No. 1 Backpack, Fit...",men's clothing,4,109.95,439.80
1,1,2,Mens Casual Premium Slim Fit T-Shirts,men's clothing,1,22.30,22.30
2,1,3,Mens Cotton Jacket,men's clothing,6,55.99,335.94
3,2,2,Mens Casual Premium Slim Fit T-Shirts,men's clothing,4,22.30,89.20
4,2,1,"Fjallraven - Foldsack No. 1 Backpack, Fit...",men's clothing,10,109.95,1099.50
5,2,5,John Hardy Women's Legends Naga Gold & Si...,jewelery,2,695.00,1390.00
6,3,1,"Fjallraven - Foldsack No. 1 Backpack, Fit...",men's clothing,2,109.95,219.90
7,3,9,WD 2TB Elements Portable External Hard Dr...,electronics,1,64.00,64.00


**Common mistakes:**  
- Merging on the wrong columns — `left_on='product_id'` and `right_on='id'` not both `'id'`
- Forgetting `how='left'` — without it, products with no orders disappear
- Not selecting specific columns from `df_products` — results in duplicate columns like `id_x`, `id_y`

### Task 3.3 — Revenue by category

**Business question:** which product category generates the most revenue across all orders?

In [ ]:
# ANSWER
revenue_by_category = (
    df_joined
    .groupby('category')
    .agg(
        total_revenue = ('revenue', 'sum'),
        items_sold    = ('quantity', 'sum'),
        orders        = ('cart_id', 'nunique')
    )
    .round(2)
    .sort_values('total_revenue', ascending=False)
)

print('Revenue by category:')
revenue_by_category

**Expected insight:**  
Electronics typically generates the highest revenue despite fewer units sold — because of high unit prices.  
Clothing may sell more units but at lower revenue per item.

**Good debrief question:**  
"Electronics has the highest revenue — does that mean it is the most profitable category? What other data would you need?"

---
## Reflection — expected answers

**Q1.** What would break if the API returned 1 million products?

*Expected answers:*
- The API would likely paginate — not return all 1 million at once
- Loading everything into memory at once would be slow or crash
- The loop to build records would take much longer
- You would need cloud infrastructure (BigQuery, Spark) rather than a notebook

**Q2.** What infrastructure to run this automatically every hour?

*Expected answers:*
- A scheduler — Cloud Scheduler, Airflow, cron job
- Storage to write the output — Cloud Storage, BigQuery
- Monitoring to know if it fails — logging, alerts

**Q3.** One-sentence business insight from revenue by category:

*Example answers:*
- "Electronics generates the highest revenue despite fewer transactions — suggesting high-value items drive disproportionate income"
- "The store should prioritise electronics inventory as it accounts for the largest revenue share"

**Q4.** Why would a real ecommerce company require API authentication even for public product data?

*Expected answers:*
- Rate limiting — prevent competitors scraping all their data
- Usage tracking — know which partners are using the API and how
- Monetisation — charge for high-volume access
- Security — even public data can be misused at scale

---
## Extension answers

### Extension A — Users endpoint

In [ ]:
# ANSWER: Extension A
users = requests.get(f'{BASE}/users').json()

print(f'Total users: {len(users)}')
print()

# Build users DataFrame
user_records = []
for u in users:
    user_records.append({
        'id':       u['id'],
        'name':     f"{u['name']['firstname']} {u['name']['lastname']}",
        'email':    u['email'],
        'city':     u['address']['city'],
        'domain':   u['email'].split('@')[1]
    })

df_users = pd.DataFrame(user_records)

# Most common city
print('Users by city:')
print(df_users['city'].value_counts().to_string())
print()

# Most common email domain
print('Most common email domains:')
print(df_users['domain'].value_counts().to_string())

# NOTE:
# name is nested: u['name']['firstname'] + u['name']['lastname']
# city is nested: u['address']['city']
# email domain: split on @ and take index [1]

Total users: 10

Users by city:
city
kilcoole       2
Cullman        1
San Antonio    1
san Antonio    1
el paso        1
fresno         1
mesa           1
miami          1
fort wayne     1

Most common email domains:
domain
gmail.com    10


### Extension B — Highest spending customer

In [ ]:
# ANSWER: Extension B
# Join df_joined (has user_id + revenue) to df_users (has user info)

customer_spend = (
    df_joined
    .groupby('user_id')
    .agg(
        total_spend  = ('revenue', 'sum'),
        total_orders = ('cart_id', 'nunique'),
        total_items  = ('quantity', 'sum')
    )
    .round(2)
    .reset_index()
    .sort_values('total_spend', ascending=False)
)

# Join to users table to get names
customer_spend = pd.merge(
    customer_spend,
    df_users[['id','name','city']],
    left_on  = 'user_id',
    right_on = 'id',
    how      = 'left'
)

print('Top customers by spend:')
customer_spend[['name','city','total_orders','total_items','total_spend']]

Top customers by spend:


,name,city,total_orders,total_items,total_spend
0,john doe,kilcoole,2,27,3376.74
1,don romer,San Antonio,1,5,560.00
2,kevin ryan,Cullman,2,6,460.78
3,david morrison,kilcoole,1,3,283.90
4,william hopkins,mesa,1,1,9.85


### Extension C — Query parameters

In [ ]:
# ANSWER: Extension C
# sort=desc reverses the order (by id)
# limit=5 returns only 5 products

sorted_products = requests.get(f'{BASE}/products', params={'sort': 'desc', 'limit': 5}).json()
print(f'Returned {len(sorted_products)} products (sorted descending by id):')
for p in sorted_products:
    print(f"  id={p['id']}  £{p['price']:.2f}  {p['title'][:45]}")

# Query parameters like ?sort=desc&limit=5 let you control what the API returns
# without changing the endpoint. This is how you page through large datasets.

Returned 5 products (sorted descending by id):
  id=5  £695.00  John Hardy Women's Legends Naga Gold & Silver
  id=4  £15.99  Mens Casual Slim Fit
  id=3  £55.99  Mens Cotton Jacket
  id=2  £22.30  Mens Casual Premium Slim Fit T-Shirts 
  id=1  £109.95  Fjallraven - Foldsack No. 1 Backpack, Fits 15
